# MalPDFDeepDetection — Cloud Hybrid CPU+GPU Bulk Extraction

Recommended Compute: **GPU · Xsmall (8 CPU / 32 GB RAM / 1 GPU)**.

Architecture:
- **CPU Workers (`8` processes):** Isolated `extract_pdf` (`PyMuPDF`, `qpdf`/`pikepdf`, `pypdf`, `Arlington`, `streams`, `carving`, `JS AST`) with full `45s wall / 35s CPU` timeout preserved (`include_byte_images=False` so no time/RAM wasted on unused byte rasters).
- **GPU Accelerator (`cuda:0`):** Batched `256x256` float32 Markov byte-transition matrices and batched 16-step `RWSE` `23D` heterogeneous graph projections.
- **Hygiene & Checkpoints:** Filters `.json` non-PDFs and non-dataset assets, deduplicates by SHA-256 with strict label-provenance priority, and checkpoints every `500`-PDF chunk under `outputs/bulk_extract/`.

In [ ]:
!nvidia-smi || true
!nproc
!free -h
!df -h .

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
VENV_PY = ROOT / ".venv/bin/python"
PY = str(VENV_PY if VENV_PY.is_file() else sys.executable)
if not VENV_PY.is_file():
    subprocess.check_call([PY, "-m", "pip", "install", "-q", "-e", ".[dev]", "py7zr"])
print("Working root:", ROOT, "| Python:", PY)
subprocess.check_call([PY, "scripts/cloud_bulk_extract.py", "--benchmark-only"])

In [ ]:
# Step 1: Pin & verify Arlington TSV model (commit b7d880b7)
subprocess.check_call([PY, "scripts/cloud_bulk_extract.py", "--execute-cloud-extraction", "--setup-arlington"])

In [ ]:
# Step 2: Acquire directly from verified HTTPS sources + run Hybrid CPU+GPU extraction
# Resumable by item (ephemeral_raw/done) and by 500-PDF chunk (outputs/bulk_extract/chunks).
# Optional: add "--prune-raw-after-chunk" if disk space is tight.
subprocess.check_call([
    PY, "scripts/cloud_bulk_extract.py",
    "--execute-cloud-extraction",
    "--acquire",
    "--extract",
    "--raw-dir", "outputs/ephemeral_raw",
    "--output-dir", "outputs/bulk_extract",
    "--device", "auto",
    "--chunk-size", "500",
    "--wall-seconds", "45",
    "--cpu-seconds", "35",
    "--hard-cap-gb", "20.0",
])

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("outputs/bulk_extract/final/summary.json").read_text())
print(json.dumps(summary, indent=2))